# 06. 스크래치로 고친 것은 고쳐진 것이 아니다

> 2026-09-04 · 이동원 · 결론 문서:
> [데이터파트 v3.5 변경사항](../../docs/데이터파트/version3.5/변경사항.md)

FinanceDataReader 는 액면분할은 조정하는데 **감자는 조정하지 않습니다**. v3.3(09-04 오전)이
그 17자리를 찾았고, 같은 날 **스크래치 스크립트**로 43,777행을 보정했습니다. DB 는 고쳐졌고
검사기도 통과했습니다.

그런데 **그 스크립트가 저장소에 없었습니다.** DB 를 다시 깔면 17자리가 그대로 돌아옵니다.
그리고 재 보니 그 보정 자체에 결함이 하나 있었습니다 — 이 노트북이 그것을 찾고, 규칙을
정본 코드로 옮긴 뒤, 결과를 바깥 값(KRX 등락률)으로 검증한 기록입니다.

In [1]:
import json
import sqlite3
import sys
from collections import Counter
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(ROOT))

from ingest.store.adj_price import (  # noqa: E402, I001
    CA_PRICE_TOLERANCE,
    CA_SCALE_TOLERANCE,
    SOURCE_CA_FIX,
)

pd.set_option("display.width", 150)
con = sqlite3.connect(f"file:{ROOT / 'data' / 'krx_cache.db'}?mode=ro", uri=True)
con.row_factory = sqlite3.Row
print("임계 — 원가격이 계수만큼 튀었나", CA_PRICE_TOLERANCE,
      "· FDR 이 안 폈나", CA_SCALE_TOLERANCE, "· 표시", SOURCE_CA_FIX)

임계 — 원가격이 계수만큼 튀었나 0.3 · FDR 이 안 폈나 0.1 · 표시 +ca_fix


## 1. 스크래치 보정이 남긴 것 — 22MB 원장과 결함 하나

보정 스크립트는 `reports/fix_capital_reduction_<시각>.json` 에 **어느 행을 어떻게 고쳤는지**를
남겼습니다(22MB · `.gitignore`). 자리 17개와 행 43,777개입니다.

In [2]:
원장경로 = sorted((ROOT / "reports").glob("fix_capital_reduction_*.json"))
if 원장경로:
    원장 = json.loads(원장경로[-1].read_text(encoding="utf-8"))
    print(f"{원장경로[-1].name} · {원장경로[-1].stat().st_size / 2**20:.1f} MB")
    print(f"자리 {len(원장['자리'])}개 · 행 {len(원장['행']):,}개")
    자리 = pd.DataFrame(원장["자리"])[["code", "name", "종류", "전일", "당일", "배율", "adj배"]]
    display(자리)
else:
    print("원장이 없습니다 (gitignore 라 다른 사람 작업 트리에는 없을 수 있습니다)")
    원장 = None

fix_capital_reduction_20260904T114044.json · 21.2 MB
자리 17개 · 행 43,777개


,code,name,종류,전일,당일,배율,adj배
0,001465,BYC우,구형우선주,20240416,20240417,0.100000,0.086
1,003060,에이프로젠바이오로직스,보통주,20260507,20260508,15.000001,13.865
2,004555,대우송도개발1우,구형우선주,20120511,20120514,6.974182,5.000
3,007460,에이프로젠,보통주,20260507,20260508,15.000000,15.461
4,009310,참엔지니어링,보통주,20260508,20260511,5.000001,5.005
5,009415,태영건설우,구형우선주,20240711,20240712,2.003376,2.000
6,012170,아센디오,보통주,20250305,20250306,10.000001,8.978
7,012170,아센디오,보통주,20260827,20260828,5.000001,4.953
8,017170,훈영,보통주,20110401,20110404,20.000007,20.000
9,021045,대호특수강우,신형우선주,20240924,20240925,0.500000,0.499


### 🔴 감자가 **두 번**인 종목에서 하루가 빠졌다

아센디오(012170)는 2025-03-06 에 10:1, 2026-08-28 에 5:1 감자를 겪었습니다. 가장 오래된
구간은 두 배율이 **곱해져** ×50 이 되어야 합니다.

스크래치 보정은 두 번째 자리를 처리하며 **첫 감자 당일(20250306) 하루를 빠뜨렸습니다.**
그 하루만 ×5 를 못 받아, 다음날 대비 `+312.6%` 가 됐습니다.

In [3]:
if 원장:
    아센 = [r for r in 원장["행"] if r["code"] == "012170"]
    print(f"012170 보정된 행 {len(아센):,} · factor 분포 "
          f"{dict(Counter(round(r['factor'], 4) for r in 아센))}")
    print(f"  보정 대상 날짜 {아센[0]['bas_dd']} ~ {아센[-1]['bas_dd']}")
    print(f"  🔴 20250306 이 대상에 있나: "
          f"{any(r['bas_dd'] == '20250306' for r in 아센)}  ← 빠졌다")

012170 보정된 행 4,098 · factor 분포 {50.0: 3737, 5.0: 361}
  보정 대상 날짜 20100104 ~ 20260827
  🔴 20250306 이 대상에 있나: False  ← 빠졌다


## 2. 규칙을 정본 코드로 옮긴다 — 경계를 손으로 잡지 않는다

`ingest/store/adj_price.py` 의 `scale_series` 에 ⑤단계를 넣었습니다. 앞에서부터 순회하며
자본변동 자리마다 **그 앞 구간 전체**에 부족한 배율을 곱합니다. 여러 번이면 자연히 누적되므로
아센디오 같은 실수가 나올 수 없습니다.

관문이 둘입니다 — 하나만 보면 멀쩡한 값을 망칩니다.

| 관문 | 무엇을 보나 | 왜 |
|---|---|---|
| ① 원가격 | `close` 비율이 계수만큼 튀었나 (±30%) | 재개일 계수는 **주식수 배율**에서 나오는데,
주식수가 움직여도 가격이 연속인 사건이 있다 |
| ② scale | FDR 이 심은 배율이 1 에 붙어 있나 (±10%) | FDR 이 **아무것도 안 한** 자리만 편다 |

두 관문을 실측으로 정했습니다 — 자본변동 크기의 계수가 있는 자리를 전수로 재서, 안 편
17자리와 이미 편 24자리가 어떻게 갈리는지 봤습니다.

In [4]:
# `verify_base_info.py` §9 와 같은 판정을 SQL 로 — 주식수 배율 ≥2 이고 adj 가 안 이어진 자리
후보 = pd.read_sql("""
    WITH t AS (
      SELECT code, isu_abbrv, kind_stkcert_tp_nm AS 종류, bas_dd,
             CAST(list_shrs AS INTEGER) s,
             LAG(CAST(list_shrs AS INTEGER)) OVER w p,
             LAG(bas_dd) OVER w pd
        FROM stock_base_info
      WINDOW w AS (PARTITION BY code ORDER BY bas_dd))
    SELECT code, isu_abbrv, 종류, pd 전일, bas_dd 당일, p 전주식수, s 후주식수
      FROM t WHERE p > 0 AND s > 0 AND (s * 1.0 / p >= 2 OR p * 1.0 / s >= 2)
     ORDER BY bas_dd""", con)
print(f"주식수가 2배 이상 변한 자리 {len(후보):,}")

def adj(code, day):
    r = con.execute("SELECT adj_close FROM daily_price WHERE code=? AND bas_dd=?",
                    (code, day)).fetchone()
    return r[0] if r else None

def close_(code, day):
    r = con.execute("SELECT close FROM daily_price WHERE code=? AND bas_dd=?",
                    (code, day)).fetchone()
    return r[0] if r else None

행 = []
for _, r in 후보.iterrows():
    a, b = adj(r.code, r.전일), adj(r.code, r.당일)
    ca, cb = close_(r.code, r.전일), close_(r.code, r.당일)
    if not all([a, b, ca, cb]):
        continue
    이론 = r.전주식수 / r.후주식수
    행.append({"code": r.code, "이름": r.isu_abbrv, "당일": r.당일, "이론배율": 이론,
               "원가격비": cb / ca, "adj비": b / a})
표 = pd.DataFrame(행)
# 🔴 판정이 두 단이다 (verify_base_info §9 와 같다).
#    ① 원가격이 이론 점프를 따라갔나 — 안 따라갔으면 증자·전환이라 조정 대상이 아니다
#    ② 그런데 adj 도 따라갔나 — 따라갔으면 조정이 안 된 것이다
표["가격이튀었나"] = (표["원가격비"] / 표["이론배율"] - 1).abs() <= 0.30
표["못이었나"] = 표["가격이튀었나"] & ((표["adj비"] / 표["이론배율"] - 1).abs() < 0.30)
print(f"  가격이 안 튀었다(증자·전환 등 — 조정이 필요 없다) : {int((~표['가격이튀었나']).sum())}")
print(f"  가격이 튀었고 수정주가가 이었다                   : "
      f"{int((표['가격이튀었나'] & ~표['못이었나']).sum())}")
print(f"  🔴 못 이은 자리                                   : {int(표['못이었나'].sum())}")
표[표["못이었나"]].assign(이론배율=lambda d: d["이론배율"].round(3),
                          원가격비=lambda d: d["원가격비"].round(3),
                          adj비=lambda d: d["adj비"].round(3))

주식수가 2배 이상 변한 자리 2,048


  가격이 안 튀었다(증자·전환 등 — 조정이 필요 없다) : 984
  가격이 튀었고 수정주가가 이었다                   : 1064
  🔴 못 이은 자리                                   : 0


,code,이름,당일,이론배율,원가격비,adj비,가격이튀었나,못이었나


`verify_base_info.py` §9 가 게이트로 매번 재는 값이고, **0 이어야 반출이 통과합니다.**
아래가 지금 상태입니다.

In [5]:
import subprocess

p = subprocess.run([sys.executable, str(ROOT / "scripts" / "verify_base_info.py")],
                   capture_output=True, text=True, encoding="utf-8", cwd=ROOT)
print("\n".join(p.stdout.strip().splitlines()[-8:]))
print("종료코드", p.returncode, "— 0 이면 반출 게이트 통과")


── 9. 자본변동을 수정주가가 이었나 ──
  주식수가 2배 이상 변한 자리 2,048 · 시세 없음 0
    가격이 안 튀었다(증자·전환 등 조정이 필요 없다) 984
    가격이 튀었고 수정주가가 이었다              1,064
  ✅ 수정주가가 못 이은 자리 0

── 판정 ── ✅ 이상 없음
종료코드 0 — 0 이면 반출 게이트 통과


## 3. 결과를 **바깥 값**으로 검증한다

규격 검사(§2)는 "우리 규칙끼리 맞나" 를 봅니다. 값이 옳은지는 **KRX 등락률**로만 알 수 있습니다.
수정주가로 잰 하루 수익률이 KRX 가 발표한 등락률과 같아야 합니다.

In [6]:
검증 = []
for code, 전일, 당일 in [("012170", "20250305", "20250306"),   # 10:1 감자
                        ("012170", "20250306", "20250307"),   # 🔴 스크래치가 빠뜨린 그 다음날
                        ("012170", "20260827", "20260828"),   # 5:1 감자
                        ("009415", "20200921", "20200922"),   # 인적분할 — 펴면 안 되는 자리
                        ("001465", "20240416", "20240417")]:  # 액면분할 10배
    a, b = adj(code, 전일), adj(code, 당일)
    krx = con.execute("SELECT change_rate, name FROM daily_price WHERE code=? AND bas_dd=?",
                      (code, 당일)).fetchone()
    src = con.execute("SELECT adj_source FROM daily_price WHERE code=? AND bas_dd=?",
                      (code, 당일)).fetchone()[0]
    검증.append({"종목": f"{code} {krx['name']}", "구간": f"{전일}→{당일}",
                 "adj 수익률": (b / a - 1) * 100, "KRX 등락률": krx["change_rate"],
                 "차이(%p)": (b / a - 1) * 100 - krx["change_rate"], "source": src})
d = pd.DataFrame(검증)
d.assign(**{"adj 수익률": d["adj 수익률"].round(2), "차이(%p)": d["차이(%p)"].round(2)})

,종목,구간,adj 수익률,KRX 등락률,차이(%p),source
0,012170 아센디오,20250305→20250306,-10.22,3.25,-13.47,fdr+ca_fix
1,012170 아센디오,20250306→20250307,-17.48,-17.48,-0.00,fdr+ca_fix
2,012170 아센디오,20260827→20260828,-0.94,-0.94,0.00,fdr
3,009415 태영건설우,20200921→20200922,-29.85,-29.85,0.00,fdr+ca_fix
4,001465 BYC우,20240416→20240417,-14.03,-14.03,0.00,fdr


네 자리는 KRX 등락률과 **소수점까지 같습니다.** 첫 줄(감자 당일)만 −13.47%p 벌어지는데,
그 이유가 이 방법의 한계를 정확히 보여 줍니다.

**계수를 상장주식수 배율에서 얻는데 KRX 기준가는 그와 다릅니다.** 아센디오는 주식수가
10.0000008배로 줄었지만 KRX 가 잡은 기준가는 전일종가의 **8.696배**(230 → 2,000)였습니다.
감자 비율과 기준가 비율은 같지 않습니다 — 그 차이가 그대로 남습니다.

| | 그 하루의 수익률 |
|---|---:|
| 안 편 상태 (원가격) | **+797.8%** |
| ⑤가 편 뒤 | −10.22% |
| KRX 등락률 (참값) | +3.25% |

**여전히 틀립니다.** 다만 크기가 800%p 에서 13%p 로 내려왔고, 무엇보다 **그 다음 날부터는
정확합니다**(둘째 줄 −17.48%). 자본변동 **당일 하루**는 어떤 배율을 쓰든 KRX 기준가를 모르면
맞출 수 없고, 그 하루는 `corporate_actions` 의 `capital_change` 플래그로 표시돼 있어 학습에서
가려낼 수 있습니다.

> 더 정확히 하려면 계수를 **기준가(`close - change`)** 에서 얻어야 합니다. 그런데 재개일에는
> 그 기준가를 믿을 수 없어(정지 중 종가가 무의미) 주식수 배율을 쓰는 것이고, 그 규칙은
> `adjustment_factor` 의 결정입니다 — 이 노트북의 범위 밖입니다.

## 4. 극단 수익률 32행이 0행이 됐다

v3.3 은 수정주가 하루 변화가 ±100% 를 넘는 96행 중 **32행**을 "설명 안 됨" 으로 남겼습니다.
`common/corporate_actions.py` 의 분류기를 수정주가 축에 붙여 다시 셉니다.

In [7]:
from common.corporate_actions import flag_series  # noqa: E402

달력 = [r[0] for r in con.execute(
    "SELECT bas_dd FROM trading_calendar WHERE market='ALL' ORDER BY bas_dd")]
idx = {d: i for i, d in enumerate(달력)}
상장중 = {r[0] for r in con.execute(
    "SELECT code FROM daily_price WHERE bas_dd=?", (달력[-1],))}
COLS = ("bas_dd, code, name, open, high, low, close, change, change_rate, volume, "
        "listed_shares, adj_close, adj_source")

극단 = []
for (code,) in con.execute("SELECT DISTINCT code FROM daily_price ORDER BY code"):
    rows = [dict(r) for r in con.execute(
        f"SELECT {COLS} FROM daily_price WHERE code=? ORDER BY bas_dd", (code,))]
    if len(rows) < 2:
        continue
    flags = flag_series(rows, calendar_index=idx, market_last_index=len(달력) - 1,
                        still_listed=code in 상장중, collect_start=달력[0])
    for i in range(1, len(rows)):
        a, b = rows[i - 1]["adj_close"], rows[i]["adj_close"]
        if not a or not b or a <= 0 or abs(b / a - 1) <= 1.0:
            continue
        f = flags[i]
        극단.append({"code": code, "이름": rows[i]["name"], "날짜": rows[i]["bas_dd"],
                     "변화%": (b / a - 1) * 100,
                     "플래그": "+".join(k for k, v in (
                         ("정리매매", f.liquidation), ("정지재개", f.halt_resume),
                         ("자본변동", f.capital_change), ("신규상장", f.first_listing)) if v)})
극단 = pd.DataFrame(극단)
print(f"수정주가 하루 변화 ±100% 초과 : {len(극단)}행")
display(극단["플래그"].replace("", "🔴 설명 안 됨").value_counts().to_frame("행 수"))
남은 = 극단[극단["플래그"] == ""]
print(f"🔴 설명 안 된 행 : {len(남은)}  (v3.3 에서는 32행이었다)")

수정주가 하루 변화 ±100% 초과 : 84행


,행 수
플래그,
정리매매,32
정지재개+자본변동,26
정리매매+정지재개,14
정지재개,6
정리매매+정지재개+자본변동,4
자본변동,2


🔴 설명 안 된 행 : 0  (v3.3 에서는 32행이었다)


32행이 **0행**이 됐습니다. 두 가지가 함께 작용했습니다 — 감자 보정으로 대부분이 닫혔고,
정본화 과정에서 찾은 아센디오 한 행이 마지막으로 닫혔습니다.

⚠️ 남은 84행은 "이상치" 가 아니라 **설명된 사건**입니다. 정리매매·거래정지 재개·자본변동은
실제로 그만큼 움직인 날이고, 학습에서 **덜어낼지**는 피처 층(`features/`)의 판단입니다.
수집 층은 자르지 않습니다([품질 규약 §4.2](../../docs/데이터파트/version3.5/데이터_품질_규약.md)).

## 5. 오늘 배운 것

**스크래치로 고친 것은 고쳐진 것이 아니다.** DB 는 맞았고 검사기도 통과했지만, 만든 방법이
저장소에 없으면 다시 깔 때 되돌아온다. 하루에 같은 모양을 셋 만났다 — 반출 스크립트,
카이제곱 코드, 감자 보정.

**경계를 손으로 잡으면 하나를 빠뜨린다.** 자리 17개를 손으로 훑으며 "그 앞 전체" 를 잡다가
두 번째 감자에서 첫 감자 당일을 놓쳤다. 규칙 하나로 앞에서부터 전파하면 그 실수가 나올 수 없다.

**계수를 믿기 전에 원가격으로 검산한다.** 재개일 계수는 주식수 배율에서 나오는데, 인적분할
(009415)과 출자전환(009410)은 주식수가 크게 움직여도 가격이 연속이다. 그 자리를 펴면 멀쩡한
값을 망친다 — 실제로 첫 구현이 그렇게 했고, 관문을 하나 더 두어 막았다.

**임계는 실측이 정한다.** 안 편 17자리는 `|scale비 - 1|` 이 전부 0.011 이하였고 이미 편
자리는 0.24 이상이었다. 20배 넘게 벌어져 있어 임계를 어디에 두든 답이 같다 — 그걸 확인하고
나서 숫자를 적었다.

> [TIL 2026-09-04](../../docs/TIL/이동원/2026-09-04-스크래치로-고친-것은-고쳐진-것이-아니다.md)